In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import laplacian
import pandas as pd
import concurrent.futures

import torch
import torch.nn as nn
import torch_geometric as tg
import torch.nn.init as init
import torch.nn.functional as F
from torch_geometric.utils import softmax
import torch.nn.utils as utils
from torch.autograd import Variable
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

from GraphNetConv import GraphNetConv
# from regularizer_functions import no_reg, gtv, fkpp, esm

/Users/vibhabalaji/anaconda3/envs/GAN_Dec2022_DIGxgraph/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
edge_index = torch.load('../data/66 ROIs processing HABS_ADNI/edge_index.pt')
habs_edge_weight = torch.load('../data/66 ROIs processing HABS_ADNI/habs_edge_weight.pt')
adni_edge_weight = torch.load('../data/66 ROIs processing HABS_ADNI/adni_edge_weight.pt')
edge_weight = torch.load('../data/66 ROIs processing HABS_ADNI/implemented_edge_weight.pt')

In [32]:
cohort = 'adni'

train_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/{cohort}_train_data_objects.pt')
val_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/{cohort}_val_data_objects.pt')
test_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/{cohort}_test_data_objects.pt')

In [33]:
A = torch.sparse_coo_tensor(edge_index, edge_weight[:,0], size = (66,66)).to_dense()
D = torch.diag(A.sum(dim=1))
L = D - A
L.to_sparse()
L = L.to(torch.float32)

In [34]:
from typing import List, Optional, Tuple, Union, Final

import torch.nn.functional as F
import torch
from torch import Tensor

from torch_geometric import EdgeIndex
from torch_geometric.nn.aggr import Aggregation, MultiAggregation
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.nn.dense.linear import Linear
from torch_geometric.typing import Adj, OptPairTensor, Size, SparseTensor, OptTensor
from torch_geometric.utils import spmm


class GraphNetConv(MessagePassing):
    SUPPORTS_FUSED_EDGE_INDEX: Final[bool] = False
        
    def __init__(
        self,
        in_channels: Union[int, Tuple[int, int]],
        out_channels: int,
        aggr: Optional[Union[str, List[str], Aggregation]] = "mean",
        normalize: bool = False,
        root_weight: bool = True,
#         global_weight: bool = False,
        project: bool = False,
        bias: bool = True,
        **kwargs,
    ):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.normalize = normalize
        self.root_weight = root_weight
#         self.global_weight = global_weight
        self.project = project

        if isinstance(in_channels, int):
            in_channels = (in_channels, in_channels)

        if aggr == 'lstm':
            kwargs.setdefault('aggr_kwargs', {})
            kwargs['aggr_kwargs'].setdefault('in_channels', in_channels[0])
            kwargs['aggr_kwargs'].setdefault('out_channels', in_channels[0])

#         if aggr == 'mlp':
#             kwargs.setdefault('aggr_kwargs', {})
#             kwargs['aggr_kwargs'].setdefault('in_channels', in_channels[0])
#             kwargs['aggr_kwargs'].setdefault('out_channels', in_channels[0])
#             kwargs['aggr_kwargs'].setdefault('max_num_elements', in_channels[0]*2)
#             kwargs['aggr_kwargs'].setdefault('num_layers', 2)
#             kwargs['aggr_kwargs'].setdefault('hidden_channels', in_channels[0]*2)
            
        super().__init__(aggr=aggr, **kwargs)

        if self.project:
            if in_channels[0] <= 0:
                raise ValueError(f"'{self.__class__.__name__}' does not "
                                 f"support lazy initialization with "
                                 f"`project=True`")
            self.lin = Linear(in_channels[0], in_channels[0], bias=True)

        if isinstance(self.aggr_module, MultiAggregation):
            aggr_out_channels = self.aggr_module.get_out_channels(
                in_channels[0])
        else:
            aggr_out_channels = in_channels[0]

        self.lin_l = Linear(aggr_out_channels, out_channels, bias=bias)
        if self.root_weight:
            self.lin_r = Linear(in_channels[1], out_channels, bias=False)
#         if self.global_weight:
#             self.lin_g = Linear(1, out_channels, bias = False)

        self.reset_parameters()

    def reset_parameters(self):
        super().reset_parameters()
        if self.project:
            self.lin.reset_parameters()
        self.lin_l.reset_parameters()
        if self.root_weight:
            self.lin_r.reset_parameters()
#         if self.global_weight:
#             self.lin_g.reset_parameters()

    def forward(
        self,
        x: Union[Tensor, OptPairTensor],
        edge_index: Adj,
        edge_weight: OptTensor = None,
#         u: OptTensor = None,
        size: Size = None,
    ) -> Tensor:

        if isinstance(x, Tensor):
            x = (x, x)

        if self.project and hasattr(self, 'lin'):
            x = (self.lin(x[0]).relu(), x[1])

#         propagate_type: (x: OptPairTensor, edge_weight: OptTensor)
        out = self.propagate(edge_index, x=x, edge_weight=edge_weight, size=size)
        out = self.lin_l(out)

        x_r = x[1]
        if self.root_weight and x_r is not None:
            out = out + self.lin_r(x_r)
            
#         if self.global_weight:
#             out = out + self.lin_g(u)

        if self.normalize:
            out = F.normalize(out, p=2., dim=-1)

        return out

    def message(self, x_j: Tensor, edge_weight: OptTensor) -> Tensor:
        if edge_weight is not None:
#             edge_weight = edge_weight.view(-1,1)
#             print("inside message")
            x_j = edge_weight * x_j
            x_j = x_j.to(torch.float32)
        return x_j

#     def message_and_aggregate(self, adj_t: Adj, x: OptPairTensor) -> Tensor:
#         print("inside message and aggregate")
#         if isinstance(adj_t, SparseTensor):
            
#             adj_t = adj_t.set_value(None, layout=None)
#         return spmm(adj_t, x[0], reduce=self.aggr)

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}({self.in_channels}, '
                f'{self.out_channels}, aggr={self.aggr})')

In [35]:
class ROIMessagePassing(MessagePassing):
    def __init__(self, in_channels, out_channels, n_rois=66):
        super().__init__(aggr='add')  # or 'mean', 'max'
        self.n_rois = n_rois

        # ROI-specific weights (shared across subjects)
        self.self_weights = nn.Parameter(torch.randn(n_rois, in_channels, out_channels))
        self.self_bias = nn.Parameter(torch.zeros(n_rois, out_channels))

        # For neighbor messages
        self.msg_linear = nn.Linear(in_channels, out_channels)
        self.act = nn.LeakyReLU(negative_slope=0.01)

        self.reset_parameters()


    def forward(self, x, edge_index, edge_weight=None):
        # x: [N, in_channels] where N = num_nodes = n_rois

        # Self transformation with ROI-specific weights
        out = torch.stack([
            torch.matmul(x[i], self.self_weights[i]) + self.self_bias[i]
            for i in range(self.n_rois)
        ], dim=0)
        

        # Message passing from neighbors
        out += self.propagate(edge_index, x=x, edge_weight=edge_weight)
        out = self.act(out)

        return out

    def message(self, x_j, edge_weight):
        msg = self.msg_linear(x_j)
        return edge_weight.view(-1, 1) * msg if edge_weight is not None else msg

In [36]:
class EncodersParallel(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, n_rois=66):
        super().__init__()
        self.n_rois = n_rois
        self.weight1 = nn.Parameter(torch.randn(1, n_rois, in_channels, hidden_channels))
        self.weight2 = nn.Parameter(torch.randn(1, n_rois, hidden_channels, hidden_channels))
        self.weight3 = nn.Parameter(torch.randn(1, n_rois, hidden_channels, out_channels))
        
        self.bias1 = nn.Parameter(torch.randn(1, n_rois, 1, hidden_channels)) 
        self.bias2 = nn.Parameter(torch.randn(1, n_rois, 1, hidden_channels)) 
        self.bias3 = nn.Parameter(torch.randn(1, n_rois, 1, out_channels))  

        self.act = nn.LeakyReLU(negative_slope=0.01)

        self.reset_parameters()

    def reset_parameters(self):
        torch.nn.init.kaiming_normal_(self.weight1, nonlinearity='leaky_relu')
        torch.nn.init.kaiming_normal_(self.weight2, nonlinearity='leaky_relu')
        torch.nn.init.kaiming_normal_(self.weight3, nonlinearity='leaky_relu')

        torch.nn.init.kaiming_normal_(self.bias1, nonlinearity='leaky_relu')
        torch.nn.init.kaiming_normal_(self.bias2, nonlinearity='leaky_relu')
        torch.nn.init.kaiming_normal_(self.bias3, nonlinearity='leaky_relu')

    def forward(self, x, edge_index, edge_weight):
        batch_size = x.shape[0] // self.n_rois
        x = x.view(batch_size, self.n_rois, -1)
        x = x.unsqueeze(2)

        x = torch.matmul(x, self.weight1) + self.bias1
        x = self.act(x)

        x = torch.matmul(x, self.weight2) + self.bias2
        x = self.act(x)

        x = torch.matmul(x, self.weight3) + self.bias3
        return x.view(batch_size * self.n_rois, -1)

In [77]:
n_rois = 66

def no_reg(inputs, outputs, edge_index):
    return torch.tensor([0.0])

def fkpp(inputs, outputs, edge_index):
    batch_size = int(len(inputs)/n_rois)
    inputs = inputs.view(batch_size, n_rois, -1)
    outputs = outputs.view(batch_size, n_rois, -1)
    loss_fkpp = []
    sigma, beta = 0.01, 0.01
    for inp, outp in zip(inputs, outputs):
        x = inp.clone()
        source_term = sigma * x * (1-x)
        diffusion_term = -beta * torch.sparse.mm(L, x)
        x = x + diffusion_term + source_term
        loss_fkpp.append(nn.functional.mse_loss(x, x+outp))
    return torch.stack(loss_fkpp).mean()

def esm(inputs, outputs, edge_index, epsilon, delta):
    epsilon = 0.001 + torch.exp(epsilon)
    delta = 0.001 + torch.exp(delta)
    batch_size = int(len(inputs)/n_rois)
    inputs = inputs.view(batch_size, n_rois, -1)
    outputs = outputs.view(batch_size, n_rois, -1)
    loss_esm = []
#     epsilon, delta = 0.01, 0.01
    for inp, outp in zip(inputs, outputs):
        x = inp.clone()
        p = torch.sigmoid(x)
        G = torch.full((n_rois, n_rois), 1.0)
        G.fill_diagonal_(1.0)
        B = torch.mul(A, G).to(torch.float32)
        infection_term = torch.mul((1-p), torch.mul(torch.matmul(
            B,(1-torch.exp(-epsilon*p))),p))
        clearance_term = torch.mul(torch.exp(-delta*p),p)
        p = p + infection_term + clearance_term
        loss_esm.append(nn.functional.mse_loss(p, torch.sigmoid(outp)))
    return torch.stack(loss_esm).mean()

In [78]:
class GraphNet(nn.Module):
    def __init__(self):
        super(GraphNet, self).__init__()
        
        self.layer1 = GraphNetConv(1, 32, aggr='PowerMeanAggregation', flow="target_to_source")

        self.layer2 = GraphNetConv(32, 64, aggr='PowerMeanAggregation', flow="target_to_source")

        self.layer3 = GraphNetConv(64, 64, aggr='PowerMeanAggregation', flow="target_to_source")
        self.layer6 = GraphNetConv(64, 64, aggr='PowerMeanAggregation', flow="target_to_source")
        self.layer7 = GraphNetConv(64, 32, aggr='PowerMeanAggregation', flow="target_to_source")
#         self.layer8 = GraphNetConv(128, 64, aggr='PowerMeanAggregation', flow="target_to_source")
#         self.layer9 = GraphNetConv(64, 32, aggr='PowerMeanAggregation', flow="target_to_source")
        
        self.layer4 = GraphNetConv(32, 16, aggr='PowerMeanAggregation', flow="target_to_source")
        
        self.layer5 = GraphNetConv(16, 1, aggr='PowerMeanAggregation', flow="target_to_source")

        self.act = nn.LeakyReLU(negative_slope=0.01)
#         self.reset_parameters()  

#     def reset_parameters(self):
#         self.layer1.reset_parameters()
#         self.layer2.reset_parameters()
#         self.layer3.reset_parameters()
# #         self.layer6.reset_parameters()
# #         self.layer7.reset_parameters()
#         self.layer4.reset_parameters()
#         self.layer5.reset_parameters()

    def forward(self, x, edge_index, edge_weight):
        x = x.to(torch.float32)
        edge_weight = edge_weight.to(torch.float32)
        
        x = self.layer1(x, edge_index, edge_weight)                            
        x = self.act(x)

        x = self.layer2(x, edge_index, edge_weight)                            
        x = self.act(x)
        
        x = self.layer3(x, edge_index, edge_weight)                            
        x = self.act(x)
        x = self.layer6(x, edge_index, edge_weight)                            
        x = self.act(x)
        x = self.layer7(x, edge_index, edge_weight)                            
        x = self.act(x)
#         x = self.layer8(x, edge_index, edge_weight)                            
#         x = self.act(x)
#         x = self.layer9(x, edge_index, edge_weight)                            
#         x = self.act(x)
        
        x = self.layer4(x, edge_index, edge_weight)                            
        x = self.act(x)

        x = self.layer5(x, edge_index, edge_weight)
        return x

In [79]:
class GraphNet_ED(nn.Module):
    def __init__(self):
        super(GraphNet_ED, self).__init__()
        
        self.layer1 = GraphNetConv(1, 4, aggr='PowerMeanAggregation', flow="target_to_source")

#         self.layer2 = GraphNetConv(4, 4, aggr='PowerMeanAggregation', flow="target_to_source")

#         self.layer3 = GraphNetConv(32, 32, aggr='PowerMeanAggregation', flow="target_to_source")
#         self.layer6 = GraphNetConv(64, 64, aggr='PowerMeanAggregation', flow="target_to_source")
#         self.layer7 = GraphNetConv(64, 32, aggr='PowerMeanAggregation', flow="target_to_source")
        
#         self.layer4 = GraphNetConv(16, 16, aggr='PowerMeanAggregation', flow="target_to_source")
        
        self.layer5 = GraphNetConv(4, 1, aggr='PowerMeanAggregation', flow="target_to_source")

        self.act = nn.LeakyReLU(negative_slope=0.01)
#         self.reset_parameters()  

#     def reset_parameters(self):
#         self.layer1.reset_parameters()
# #         self.layer2.reset_parameters()
# #         self.layer3.reset_parameters()
# #         self.layer4.reset_parameters()
#         self.layer5.reset_parameters()

    def forward(self, x, edge_index, edge_weight):
        x = x.to(torch.float32)
        edge_weight = edge_weight.to(torch.float32)
        
        x = self.layer1(x, edge_index, edge_weight)                            
        x = self.act(x)

#         x = self.layer2(x, edge_index, edge_weight)                            
#         x = self.act(x)
        
#         x = self.layer3(x, edge_index, edge_weight)                            
#         x = self.act(x)
#         x = self.layer6(x, edge_index, edge_weight)                            
#         x = self.act(x)
#         x = self.layer7(x, edge_index, edge_weight)                            
#         x = self.act(x)
        
#         x = self.layer4(x, edge_index, edge_weight)                            
#         x = self.act(x)

        x = self.layer5(x, edge_index, edge_weight)
        return x

In [184]:
## training

batch_size = 8
folder = cohort

train_loader = DataLoader(train_data, batch_size = batch_size, 
                          shuffle = True, drop_last = True)
val_loader = DataLoader(val_data, batch_size = batch_size, 
                        shuffle = False, drop_last = False)
test_loader = DataLoader(test_data, batch_size = batch_size, 
                         shuffle = False, drop_last = False)

loss_function = torch.nn.MSELoss()

regularizer = no_reg
savename = 'no_reg'
reg_weight = 1e-1

num_epochs = 5000
min_val_loss = 10000.0

In [185]:
# encoder = ROIMessagePassing(in_channels=1, out_channels=1, n_rois=66)
# model = GraphNet()
# decoder = ROIMessagePassing(in_channels=1, out_channels=1, n_rois=66)

encoder = GraphNet_ED()
model = GraphNet()
decoder = GraphNet_ED()

for param in model.parameters():
    param.requires_grad = True

In [186]:
### TEMPORARY -try on sample
sample = train_data[0]
# for data in train_loader:
#     sample = data
z_x = encoder(sample.x, sample.edge_index, sample.edge_weight)
z_yhat = model(z_x, sample.edge_index, sample.edge_weight)
yhat = decoder(z_yhat, sample.edge_index, sample.edge_weight)

In [187]:
def run_epoch(loader, is_training):
    total_loss, total_primary = 0.0, 0.0
    outputs = []
    
    if is_training:
        model.train(); encoder.train(); decoder.train()
    else:
        model.eval(); encoder.eval(); decoder.eval()
    
    for data in loader:
#         yhat = model(data.x)
#         yhat = model(data.x, data.edge_index, edge_weight = data.edge_weight)
        batch_len = len(data.x) // 66
        z_x = encoder(data.x, data.edge_index, data.edge_weight+1e-6)
        z_yhat = model(z_x, data.edge_index, data.edge_weight+1e-6)
        yhat = decoder(z_yhat, data.edge_index, data.edge_weight+1e-6)
        
        primary_loss = loss_function(yhat*10, data.y*10)
#         var_loss = (yhat.std(dim=0) - data.y.std(dim=0)).pow(2).mean()
#         primary_loss = primary_loss + 0.01 * var_loss

#         reg_loss = regularizer(data.x, yhat, data.edge_index.to(torch.float32))
        reg_loss = regularizer(z_x, z_yhat, data.edge_index.to(torch.float32))
        loss = primary_loss + reg_weight * reg_loss 
        
        if is_training:
            optimizer.zero_grad()
            loss.backward()
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             torch.nn.utils.clip_grad_norm_(encoder.parameters(), max_norm=1.0)
#             torch.nn.utils.clip_grad_norm_(decoder.parameters(), max_norm=1.0)
            optimizer.step()
            
        
        total_loss += (loss.item()*batch_len)
        total_primary += (primary_loss.item()*batch_len)
        outputs.append(yhat.detach())
#         outputs.append(yhat)
        
    return total_loss, total_primary, outputs

In [188]:
optimizer = torch.optim.Adam([
    {'params': encoder.parameters()},
    {'params': model.parameters()},
    {'params': decoder.parameters()}
], lr = 1e-3, weight_decay = 0)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,
#                                      mode='min', factor=0.5, patience=20)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.9,       # gentler reduction
    patience=50,      # a bit quicker to respond
    cooldown=5,       # prevent repeated triggering
    min_lr=1e-5,      # avoids vanishing LR
    verbose=True
)


In [189]:
for epoch in range(num_epochs):
    is_training = True
    train_loss, train_primary, train_outputs = run_epoch(train_loader, is_training = True)
    train_loss, train_primary = train_loss/len(train_data), train_primary/len(train_data)
    
    torch.save(encoder.state_dict(), 
           f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_encoder_state.pth')
    torch.save(model.state_dict(), 
           f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_model_state.pth')
    torch.save(decoder.state_dict(), 
           f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_decoder_state.pth')
    
    is_training = False
    val_loss, val_primary, val_outputs = run_epoch(val_loader, is_training = False)
    val_loss, val_primary = val_loss/len(val_data), val_primary/len(val_data)
    
    scheduler.step(train_loss)
    
    print(f"Epoch: {epoch}/{num_epochs} Train Loss: {train_loss:.6f}, "
          f"Train MSE: {train_primary:.6f} "
          f"| Val Loss: {val_loss:.6f}, "
          f"Val MSE: {val_primary:.6f}")

    if val_loss < min_val_loss:
        print("Saved best model")
        min_val_loss = val_loss
        best_encoder_state = encoder.state_dict()
        best_model_state = model.state_dict()
        best_decoder_state = decoder.state_dict()

Epoch: 0/5000 Train Loss: 2.676651, Train MSE: 2.676651 | Val Loss: 0.770831, Val MSE: 0.770831
Saved best model
Epoch: 1/5000 Train Loss: 1.210564, Train MSE: 1.210564 | Val Loss: 0.811481, Val MSE: 0.811481
Epoch: 2/5000 Train Loss: 1.208192, Train MSE: 1.208192 | Val Loss: 0.792830, Val MSE: 0.792830
Epoch: 3/5000 Train Loss: 1.204582, Train MSE: 1.204582 | Val Loss: 0.765213, Val MSE: 0.765213
Saved best model
Epoch: 4/5000 Train Loss: 1.213812, Train MSE: 1.213812 | Val Loss: 0.783599, Val MSE: 0.783599
Epoch: 5/5000 Train Loss: 1.210868, Train MSE: 1.210868 | Val Loss: 0.762371, Val MSE: 0.762371
Saved best model
Epoch: 6/5000 Train Loss: 1.221570, Train MSE: 1.221570 | Val Loss: 0.791985, Val MSE: 0.791985
Epoch: 7/5000 Train Loss: 1.215310, Train MSE: 1.215310 | Val Loss: 0.763793, Val MSE: 0.763793
Epoch: 8/5000 Train Loss: 1.203071, Train MSE: 1.203071 | Val Loss: 0.764041, Val MSE: 0.764041
Epoch: 9/5000 Train Loss: 1.202177, Train MSE: 1.202177 | Val Loss: 0.759247, Val MSE

Epoch: 83/5000 Train Loss: 1.196985, Train MSE: 1.196985 | Val Loss: 0.765123, Val MSE: 0.765123
Epoch: 84/5000 Train Loss: 1.195781, Train MSE: 1.195781 | Val Loss: 0.756911, Val MSE: 0.756911
Epoch: 85/5000 Train Loss: 1.197063, Train MSE: 1.197063 | Val Loss: 0.763511, Val MSE: 0.763511
Epoch: 86/5000 Train Loss: 1.199215, Train MSE: 1.199215 | Val Loss: 0.771367, Val MSE: 0.771367
Epoch: 87/5000 Train Loss: 1.197128, Train MSE: 1.197128 | Val Loss: 0.765007, Val MSE: 0.765007
Epoch: 88/5000 Train Loss: 1.194524, Train MSE: 1.194524 | Val Loss: 0.755704, Val MSE: 0.755704
Saved best model
Epoch: 89/5000 Train Loss: 1.196280, Train MSE: 1.196280 | Val Loss: 0.767565, Val MSE: 0.767565
Epoch: 90/5000 Train Loss: 1.200398, Train MSE: 1.200398 | Val Loss: 0.789073, Val MSE: 0.789073
Epoch: 91/5000 Train Loss: 1.195196, Train MSE: 1.195196 | Val Loss: 0.755993, Val MSE: 0.755993
Epoch: 92/5000 Train Loss: 1.199041, Train MSE: 1.199041 | Val Loss: 0.759490, Val MSE: 0.759490
Epoch: 93/500

Epoch: 163/5000 Train Loss: 1.191256, Train MSE: 1.191256 | Val Loss: 0.754085, Val MSE: 0.754085
Saved best model
Epoch: 164/5000 Train Loss: 1.195889, Train MSE: 1.195889 | Val Loss: 0.754604, Val MSE: 0.754604
Epoch: 165/5000 Train Loss: 1.191582, Train MSE: 1.191582 | Val Loss: 0.783784, Val MSE: 0.783784
Epoch: 166/5000 Train Loss: 1.198278, Train MSE: 1.198278 | Val Loss: 0.759543, Val MSE: 0.759543
Epoch: 167/5000 Train Loss: 1.195053, Train MSE: 1.195053 | Val Loss: 0.784216, Val MSE: 0.784216
Epoch: 168/5000 Train Loss: 1.197385, Train MSE: 1.197385 | Val Loss: 0.757898, Val MSE: 0.757898
Epoch: 169/5000 Train Loss: 1.193549, Train MSE: 1.193549 | Val Loss: 0.765827, Val MSE: 0.765827
Epoch: 170/5000 Train Loss: 1.194457, Train MSE: 1.194457 | Val Loss: 0.759546, Val MSE: 0.759546
Epoch: 171/5000 Train Loss: 1.193291, Train MSE: 1.193291 | Val Loss: 0.754310, Val MSE: 0.754310
Epoch: 172/5000 Train Loss: 1.191765, Train MSE: 1.191765 | Val Loss: 0.756224, Val MSE: 0.756224
Epo

Epoch: 245/5000 Train Loss: 1.182862, Train MSE: 1.182862 | Val Loss: 0.747385, Val MSE: 0.747385
Epoch: 246/5000 Train Loss: 1.194291, Train MSE: 1.194291 | Val Loss: 0.753542, Val MSE: 0.753542
Epoch: 247/5000 Train Loss: 1.182513, Train MSE: 1.182513 | Val Loss: 0.749646, Val MSE: 0.749646
Epoch: 248/5000 Train Loss: 1.182211, Train MSE: 1.182211 | Val Loss: 0.746214, Val MSE: 0.746214
Saved best model
Epoch: 249/5000 Train Loss: 1.186162, Train MSE: 1.186162 | Val Loss: 0.749937, Val MSE: 0.749937
Epoch: 250/5000 Train Loss: 1.183614, Train MSE: 1.183614 | Val Loss: 0.757262, Val MSE: 0.757262
Epoch: 251/5000 Train Loss: 1.187009, Train MSE: 1.187009 | Val Loss: 0.759520, Val MSE: 0.759520
Epoch: 252/5000 Train Loss: 1.183120, Train MSE: 1.183120 | Val Loss: 0.746763, Val MSE: 0.746763
Epoch: 253/5000 Train Loss: 1.184603, Train MSE: 1.184603 | Val Loss: 0.761211, Val MSE: 0.761211
Epoch: 254/5000 Train Loss: 1.179527, Train MSE: 1.179527 | Val Loss: 0.747470, Val MSE: 0.747470
Epo

Epoch: 329/5000 Train Loss: 1.170841, Train MSE: 1.170841 | Val Loss: 0.751136, Val MSE: 0.751136
Epoch: 330/5000 Train Loss: 1.169513, Train MSE: 1.169513 | Val Loss: 0.745703, Val MSE: 0.745703
Epoch: 331/5000 Train Loss: 1.170503, Train MSE: 1.170503 | Val Loss: 0.747560, Val MSE: 0.747560
Epoch: 332/5000 Train Loss: 1.171125, Train MSE: 1.171125 | Val Loss: 0.746841, Val MSE: 0.746841
Epoch: 333/5000 Train Loss: 1.173958, Train MSE: 1.173958 | Val Loss: 0.747970, Val MSE: 0.747970
Epoch: 334/5000 Train Loss: 1.169979, Train MSE: 1.169979 | Val Loss: 0.753270, Val MSE: 0.753270
Epoch: 335/5000 Train Loss: 1.167847, Train MSE: 1.167847 | Val Loss: 0.755434, Val MSE: 0.755434
Epoch: 336/5000 Train Loss: 1.165300, Train MSE: 1.165300 | Val Loss: 0.744797, Val MSE: 0.744797
Epoch: 337/5000 Train Loss: 1.168347, Train MSE: 1.168347 | Val Loss: 0.757795, Val MSE: 0.757795
Epoch: 338/5000 Train Loss: 1.166793, Train MSE: 1.166793 | Val Loss: 0.750405, Val MSE: 0.750405
Epoch: 339/5000 Trai

...

Epoch: 4630/5000 Train Loss: 0.292219, Train MSE: 0.292219 | Val Loss: 1.399188, Val MSE: 1.399188
Epoch: 4631/5000 Train Loss: 0.293122, Train MSE: 0.293122 | Val Loss: 1.388365, Val MSE: 1.388365
Epoch: 4632/5000 Train Loss: 0.294480, Train MSE: 0.294480 | Val Loss: 1.396956, Val MSE: 1.396956
Epoch: 4633/5000 Train Loss: 0.293804, Train MSE: 0.293804 | Val Loss: 1.381874, Val MSE: 1.381874
Epoch: 4634/5000 Train Loss: 0.293210, Train MSE: 0.293210 | Val Loss: 1.372262, Val MSE: 1.372262
Epoch: 4635/5000 Train Loss: 0.296431, Train MSE: 0.296431 | Val Loss: 1.380159, Val MSE: 1.380159
Epoch: 4636/5000 Train Loss: 0.295006, Train MSE: 0.295006 | Val Loss: 1.385643, Val MSE: 1.385643
Epoch: 4637/5000 Train Loss: 0.295094, Train MSE: 0.295094 | Val Loss: 1.385697, Val MSE: 1.385697
Epoch: 4638/5000 Train Loss: 0.294530, Train MSE: 0.294530 | Val Loss: 1.366769, Val MSE: 1.366769
Epoch: 4639/5000 Train Loss: 0.293671, Train MSE: 0.293671 | Val Loss: 1.369843, Val MSE: 1.369843
Epoch: 464

Epoch: 4711/5000 Train Loss: 0.293004, Train MSE: 0.293004 | Val Loss: 1.380851, Val MSE: 1.380851
Epoch: 4712/5000 Train Loss: 0.292142, Train MSE: 0.292142 | Val Loss: 1.381354, Val MSE: 1.381354
Epoch: 4713/5000 Train Loss: 0.292036, Train MSE: 0.292036 | Val Loss: 1.395739, Val MSE: 1.395739
Epoch: 4714/5000 Train Loss: 0.291670, Train MSE: 0.291670 | Val Loss: 1.378817, Val MSE: 1.378817
Epoch: 4715/5000 Train Loss: 0.292389, Train MSE: 0.292389 | Val Loss: 1.386162, Val MSE: 1.386162
Epoch: 4716/5000 Train Loss: 0.290792, Train MSE: 0.290792 | Val Loss: 1.385538, Val MSE: 1.385538
Epoch: 4717/5000 Train Loss: 0.290961, Train MSE: 0.290961 | Val Loss: 1.378389, Val MSE: 1.378389
Epoch: 4718/5000 Train Loss: 0.293030, Train MSE: 0.293030 | Val Loss: 1.391547, Val MSE: 1.391547
Epoch: 4719/5000 Train Loss: 0.294160, Train MSE: 0.294160 | Val Loss: 1.383849, Val MSE: 1.383849
Epoch: 4720/5000 Train Loss: 0.294186, Train MSE: 0.294186 | Val Loss: 1.370432, Val MSE: 1.370432
Epoch: 472

Epoch: 4792/5000 Train Loss: 0.291919, Train MSE: 0.291919 | Val Loss: 1.392887, Val MSE: 1.392887
Epoch: 4793/5000 Train Loss: 0.293544, Train MSE: 0.293544 | Val Loss: 1.382217, Val MSE: 1.382217
Epoch: 4794/5000 Train Loss: 0.298568, Train MSE: 0.298568 | Val Loss: 1.390051, Val MSE: 1.390051
Epoch: 4795/5000 Train Loss: 0.292524, Train MSE: 0.292524 | Val Loss: 1.405742, Val MSE: 1.405742
Epoch: 4796/5000 Train Loss: 0.288920, Train MSE: 0.288920 | Val Loss: 1.396834, Val MSE: 1.396834
Epoch: 4797/5000 Train Loss: 0.288827, Train MSE: 0.288827 | Val Loss: 1.382926, Val MSE: 1.382926
Epoch: 4798/5000 Train Loss: 0.287854, Train MSE: 0.287854 | Val Loss: 1.394833, Val MSE: 1.394833
Epoch: 4799/5000 Train Loss: 0.289075, Train MSE: 0.289075 | Val Loss: 1.411285, Val MSE: 1.411285
Epoch: 4800/5000 Train Loss: 0.288178, Train MSE: 0.288178 | Val Loss: 1.394902, Val MSE: 1.394902
Epoch: 4801/5000 Train Loss: 0.287753, Train MSE: 0.287753 | Val Loss: 1.395321, Val MSE: 1.395321
Epoch: 480

Epoch: 4873/5000 Train Loss: 0.286451, Train MSE: 0.286451 | Val Loss: 1.401603, Val MSE: 1.401603
Epoch: 4874/5000 Train Loss: 0.286297, Train MSE: 0.286297 | Val Loss: 1.396635, Val MSE: 1.396635
Epoch: 4875/5000 Train Loss: 0.286987, Train MSE: 0.286987 | Val Loss: 1.402335, Val MSE: 1.402335
Epoch: 4876/5000 Train Loss: 0.287322, Train MSE: 0.287322 | Val Loss: 1.414256, Val MSE: 1.414256
Epoch: 4877/5000 Train Loss: 0.287883, Train MSE: 0.287883 | Val Loss: 1.405230, Val MSE: 1.405230
Epoch: 4878/5000 Train Loss: 0.287952, Train MSE: 0.287952 | Val Loss: 1.404467, Val MSE: 1.404467
Epoch: 4879/5000 Train Loss: 0.286779, Train MSE: 0.286779 | Val Loss: 1.395849, Val MSE: 1.395849
Epoch: 4880/5000 Train Loss: 0.286893, Train MSE: 0.286893 | Val Loss: 1.394983, Val MSE: 1.394983
Epoch: 4881/5000 Train Loss: 0.287322, Train MSE: 0.287322 | Val Loss: 1.393684, Val MSE: 1.393684
Epoch: 4882/5000 Train Loss: 0.286682, Train MSE: 0.286682 | Val Loss: 1.402221, Val MSE: 1.402221
Epoch: 488

Epoch: 4954/5000 Train Loss: 0.287057, Train MSE: 0.287057 | Val Loss: 1.414017, Val MSE: 1.414017
Epoch: 4955/5000 Train Loss: 0.286329, Train MSE: 0.286329 | Val Loss: 1.408983, Val MSE: 1.408983
Epoch: 4956/5000 Train Loss: 0.286498, Train MSE: 0.286498 | Val Loss: 1.415178, Val MSE: 1.415178
Epoch: 4957/5000 Train Loss: 0.286108, Train MSE: 0.286108 | Val Loss: 1.402876, Val MSE: 1.402876
Epoch: 4958/5000 Train Loss: 0.286794, Train MSE: 0.286794 | Val Loss: 1.415553, Val MSE: 1.415553
Epoch: 4959/5000 Train Loss: 0.285396, Train MSE: 0.285396 | Val Loss: 1.408854, Val MSE: 1.408854
Epoch: 4960/5000 Train Loss: 0.284687, Train MSE: 0.284687 | Val Loss: 1.420673, Val MSE: 1.420673
Epoch: 4961/5000 Train Loss: 0.285231, Train MSE: 0.285231 | Val Loss: 1.412804, Val MSE: 1.412804
Epoch: 4962/5000 Train Loss: 0.285740, Train MSE: 0.285740 | Val Loss: 1.404656, Val MSE: 1.404656
Epoch: 4963/5000 Train Loss: 0.285364, Train MSE: 0.285364 | Val Loss: 1.410120, Val MSE: 1.410120
Epoch: 496

In [176]:
# do this if the learning plateaus
# for param_group in optimizer.param_groups:
#     param_group['lr'] *= 2

In [177]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name}: grad = {param.grad.norm() if param.grad is not None else 'None'}")


layer1.lin_l.weight: grad = 0.00038238195702433586
layer1.lin_l.bias: grad = 0.10114295035600662
layer1.lin_r.weight: grad = 0.02116134949028492
layer2.lin_l.weight: grad = 0.005037947557866573
layer2.lin_l.bias: grad = 0.1022074893116951
layer2.lin_r.weight: grad = 0.24568599462509155
layer3.lin_l.weight: grad = 0.004797541536390781
layer3.lin_l.bias: grad = 0.07590986788272858
layer3.lin_r.weight: grad = 0.20381788909435272
layer6.lin_l.weight: grad = 0.0031784381717443466
layer6.lin_l.bias: grad = 0.06165565922856331
layer6.lin_r.weight: grad = 0.13113552331924438
layer7.lin_l.weight: grad = 0.0065659331157803535
layer7.lin_l.bias: grad = 0.09473752975463867
layer7.lin_r.weight: grad = 0.13416950404644012
layer4.lin_l.weight: grad = 0.005291251465678215
layer4.lin_l.bias: grad = 0.07256502658128738
layer4.lin_r.weight: grad = 0.12541337311267853
layer5.lin_l.weight: grad = 0.011863041669130325
layer5.lin_l.bias: grad = 0.222447007894516
layer5.lin_r.weight: grad = 0.2466998398303985

In [253]:
epoch = 2499
best_encoder_state = torch.load(f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_encoder_state.pth')
best_model_state = torch.load(f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_model_state.pth')
best_decoder_state = torch.load(f'../results revision/{cohort}/weights/{savename}_Epoch_{epoch}_decoder_state.pth')

In [254]:
def get_outputs(data_objects):
    data_loader = DataLoader(data_objects, batch_size = batch_size, 
                             shuffle = False, drop_last = False)
    encoder.load_state_dict(best_encoder_state)
    model.load_state_dict(best_model_state)
    decoder.load_state_dict(best_decoder_state)
    model.eval()
    encoder.eval()
    decoder.eval()
    _, _, outputs = run_epoch(data_loader, is_training = False)
    return torch.split(torch.cat(outputs), 66)

In [255]:
get_outputs(train_data)[0] #predicted

tensor([[ 0.0782],
        [ 0.0531],
        [ 0.0310],
        [ 0.0232],
        [ 0.0234],
        [-0.0040],
        [-0.0166],
        [ 0.2164],
        [ 0.1018],
        [ 0.0148],
        [ 0.0774],
        [ 0.0491],
        [ 0.0219],
        [ 0.0157],
        [ 0.0459],
        [ 0.0088],
        [ 0.0131],
        [ 0.1725],
        [ 0.0146],
        [-0.0009],
        [ 0.0208],
        [ 0.0030],
        [ 0.0104],
        [-0.0148],
        [ 0.0247],
        [ 0.0079],
        [ 0.0114],
        [ 0.0024],
        [ 0.0105],
        [ 0.0116],
        [-0.0125],
        [ 0.0009],
        [ 0.0002],
        [-0.0140],
        [ 0.0099],
        [ 0.0146],
        [-0.0075],
        [ 0.0180],
        [ 0.0129],
        [ 0.0450],
        [ 0.0303],
        [ 0.0293],
        [ 0.0310],
        [ 0.0169],
        [ 0.0052],
        [ 0.0250],
        [-0.0041],
        [ 0.0312],
        [ 0.0441],
        [ 0.0123],
        [ 0.0062],
        [ 0.0311],
        [ 0.

In [256]:
train_data[0].y #ground truth

tensor([[-0.0397],
        [-0.0325],
        [-0.1047],
        [ 0.0271],
        [-0.0144],
        [-0.0307],
        [-0.0632],
        [ 0.2384],
        [ 0.0939],
        [ 0.0271],
        [ 0.1535],
        [-0.0343],
        [ 0.0849],
        [ 0.0614],
        [ 0.0036],
        [ 0.0181],
        [ 0.0795],
        [ 0.2672],
        [ 0.0108],
        [ 0.0163],
        [ 0.0686],
        [-0.0307],
        [-0.0361],
        [ 0.0163],
        [-0.0054],
        [ 0.0181],
        [-0.0379],
        [ 0.0018],
        [-0.0451],
        [ 0.0163],
        [-0.0090],
        [-0.0361],
        [ 0.0650],
        [-0.0271],
        [-0.0054],
        [ 0.0163],
        [ 0.0090],
        [-0.0506],
        [ 0.1517],
        [ 0.0578],
        [ 0.0199],
        [ 0.1101],
        [-0.0397],
        [ 0.0813],
        [ 0.0343],
        [-0.0108],
        [-0.0614],
        [ 0.0361],
        [ 0.1643],
        [ 0.0144],
        [ 0.0235],
        [ 0.0650],
        [-0.

In [257]:
len(get_outputs(train_data))

528

In [258]:
torch.save(get_outputs(train_data), f'../results revision/{cohort}/{savename}_best_train.pth')
torch.save(get_outputs(val_data), f'../results revision/{cohort}/{savename}_best_val.pth')
torch.save(get_outputs(test_data), f'../results revision/{cohort}/{savename}_best_test.pth')
torch.save(best_encoder_state, 
           f'../results revision/{cohort}/{savename}_best_encoder_state.pth')
torch.save(best_model_state, 
           f'../results revision/{cohort}/{savename}_best_model_state.pth')
torch.save(best_decoder_state, 
           f'../results revision/{cohort}/{savename}_best_decoder_state.pth')

In [132]:
### external validation

habs_train_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/habs_train_data_objects.pt')
habs_val_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/habs_val_data_objects.pt')
habs_test_data = torch.load(f'../data/66 ROIs processing HABS_ADNI/habs_test_data_objects.pt')
habs_data = habs_train_data + habs_val_data + habs_test_data
torch.save(get_outputs(train_data), f'../results revision/{cohort}/{savename}_best_habsval.pth')